# Reproducing `2510.13634v1` with `qrc_sim`

This notebook builds a local, executable case study around **"Multivariate Time Series Forecasting with Gate-Based Quantum Reservoir Computing on NISQ Hardware"** ([arXiv:2510.13634v1](https://arxiv.org/abs/2510.13634v1)) using **QRC-Lab / `qrc_sim`** ([arXiv:2602.03522v1](https://arxiv.org/abs/2602.03522v1)).

## Academic objective
Demonstrate that `2602.03522v1` is a real **academic tool**, not just a teaching demo, by encoding a published MTS-QRC workflow inside `qrc_sim` and turning paper claims into executable checks.

## What is reproduced here
- the **MTS-QRC rewind protocol** with injection and memory qubits,
- the **Lorenz-63** and **ENSO** datasets used in the paper,
- the paper's key **hyperparameter trends**,
- the **connectivity-depth tradeoff** that motivates Opt-NN-TFI,
- the **forecast quality** for the best local configurations,
- and a **hardware-proxy SVD study** inspired by the paper's IBM Heron analysis.

## Important scope note
This notebook is designed for **local reproducibility on a workstation**. It therefore uses:
- exact statevector simulation for the core reproduction,
- transpilation-based resource estimates instead of queueing a real QPU,
- and a shot/depolarizing proxy instead of calling IBM hardware.

The goal is not to impersonate the QPU run, but to show that `qrc_sim` can serve as an **academic scaffold** for article-oriented investigation.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from qrc_sim.mts_qrc_reproduction import (
    PAPER_2510_URL,
    QRCLAB_URL,
    ReservoirConfig,
    circuit_resource_scan,
    connectivity_scan,
    draw_connectivity_bars,
    draw_heatmap,
    draw_prediction_panel,
    draw_resource_scaling,
    draw_svd_panel,
    evaluate_configuration,
    explained_variance,
    hardware_proxy_forecast,
    load_paper_datasets,
    save_figure,
    scan_memory_and_tau,
    set_plot_style,
)

set_plot_style()

ARTIFACTS = ROOT / "qrc_sim" / "artifacts" / "mts_qrc_2510"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

datasets = load_paper_datasets(length=120, burn_in=30)
lorenz = datasets["Lorenz-63"]
enso = datasets["ENSO"]

print("Paper under study:", PAPER_2510_URL)
print("Tool paper:", QRCLAB_URL)
print("Artifacts:", ARTIFACTS)
print("Datasets loaded:", list(datasets))


## Stage 1. Rebuild the paper protocol inside `qrc_sim`

The paper defines an MTS-QRC pipeline with:
- 3 injection qubits, one per input variable,
- multiple memory qubits per variable,
- Trotterized TFI evolution,
- a 10-step rewind window,
- and a linear readout trained for next-step forecasting.

In this repository, that workflow is implemented in `qrc_sim.mts_qrc_reproduction`, so the article logic now lives inside the toolbox itself.


In [ ]:
for dataset in [lorenz, enso]:
    print(f"\n{dataset.name}")
    print("raw shape   :", dataset.raw.shape)
    print("encoded min :", dataset.encoded.min(axis=0))
    print("encoded max :", dataset.encoded.max(axis=0))
    print("paper MSE   :", dataset.metadata["paper_mse"])

print("\nCorrectness check")
print("- Dataset dimensionality is 3 in both cases:", lorenz.raw.shape[1] == 3 and enso.raw.shape[1] == 3)
print("- Encoded data stays inside [0, 1]:", float(lorenz.encoded.min()) >= 0.0 and float(enso.encoded.max()) <= 1.0)


**Reflection**

- Correct if the data is 3-dimensional and the encoding is bounded in `[0, 1]`, because that matches the paper's injection rule.
- Not correct if either dataset breaks that range, because the `R_y(2 arcsin(sqrt(u)))` map would no longer match the article.


## Stage 2. Check the local hyperparameter landscape

Table 2 of the paper reports the best settings:
- **Lorenz-63**: 3 memory qubits per input, `tau = 1.0`
- **ENSO**: 3 memory qubits per input, `tau = 0.1`

The next scan is intentionally local and lightweight. The purpose is not a massive benchmark, but to check whether the **same academic pattern** emerges when the method is rebuilt inside `qrc_sim`.


In [ ]:
lorenz_scan = scan_memory_and_tau(
    lorenz,
    connectivity="opt_nn_tfi",
    memory_grid=(1, 2, 3, 4),
    tau_grid=(0.1, 1.0),
    seed=2,
)
enso_scan = scan_memory_and_tau(
    enso,
    connectivity="opt_nn_tfi",
    memory_grid=(1, 2, 3, 4),
    tau_grid=(0.1, 1.0),
    seed=2,
)

fig = draw_heatmap(lorenz_scan, "Lorenz-63")
save_figure(fig, ARTIFACTS / "lorenz_hyperparameters.png")
fig.show()

fig = draw_heatmap(enso_scan, "ENSO")
save_figure(fig, ARTIFACTS / "enso_hyperparameters.png")
fig.show()

best_lorenz = min(lorenz_scan, key=lambda row: row["mse"])
best_enso = min(enso_scan, key=lambda row: row["mse"])

print("Best local Lorenz config:", best_lorenz)
print("Paper-inspired Lorenz target:", {"memory_per_input": 3, "tau": 1.0})
print("Best local ENSO config  :", best_enso)
print("Paper-inspired ENSO target  :", {"memory_per_input": 3, "tau": 0.1})
print("\nCorrectness check")
print("- Lorenz local best remains in the paper's tested tau set:", best_lorenz["tau"] in {0.1, 1.0})
print("- ENSO local best remains in the paper's tested tau set  :", best_enso["tau"] in {0.1, 1.0})


**Reflection**

- Correct if the locally preferred region stays close to the paper's reported settings, even if the exact optimum moves under the lighter workstation regime.
- If one of them diverges, that is still academically useful: the notebook then becomes a reproducibility probe that surfaces sensitivity to data length, seed, and simulator assumptions.


## Stage 3. Verify the connectivity argument behind Opt-NN-TFI

The paper argues that Opt-NN-TFI preserves forecasting quality while being much more hardware-friendly than FC-TFI. We check both parts:
1. **accuracy** via a local connectivity comparison,
2. **resource scaling** via transpiled full-window circuit depth and size.


In [ ]:
lorenz_conn = connectivity_scan(lorenz, memory_per_input=3, tau=1.0, seed=2)
enso_conn = connectivity_scan(enso, memory_per_input=3, tau=0.1, seed=2)

fig = draw_connectivity_bars(lorenz_conn, "Lorenz-63")
save_figure(fig, ARTIFACTS / "lorenz_connectivity.png")
fig.show()

fig = draw_connectivity_bars(enso_conn, "ENSO")
save_figure(fig, ARTIFACTS / "enso_connectivity.png")
fig.show()

resources = circuit_resource_scan(memory_grid=(1, 2, 3, 4), tau=0.1, window_size=10)
fig = draw_resource_scaling(resources)
save_figure(fig, ARTIFACTS / "resource_scaling.png")
fig.show()

print("Lorenz connectivity rows:", lorenz_conn)
print("ENSO connectivity rows  :", enso_conn)
print("\nCorrectness check")
print("- Opt-NN-TFI stays competitive in Lorenz:", min(row["mse"] for row in lorenz_conn if row["connectivity"] == "opt_nn_tfi") <= 1.15 * min(row["mse"] for row in lorenz_conn))
print("- FC-TFI is the deepest family in transpiled circuits:", max(row["depth"] for row in resources if row["connectivity"] == "fc_tfi") > max(row["depth"] for row in resources if row["connectivity"] == "opt_nn_tfi"))


**Reflection**

- Correct if Opt-NN-TFI remains near the best MSE while FC-TFI is clearly more expensive after transpilation.
- That is exactly the kind of evidence that makes `qrc_sim` useful as an academic workbench: it lets us test **accuracy-resource tradeoffs**, not only produce curves.


## Stage 4. Reproduce the main forecasting behavior

We now use the paper-inspired best settings for each dataset and compare:
- the paper's reported MSE,
- the local baseline-copy MSE,
- and the `qrc_sim` reproduction MSE.


In [ ]:
lorenz_config = ReservoirConfig(
    dataset_name="Lorenz-63",
    connectivity="opt_nn_tfi",
    memory_per_input=3,
    tau=1.0,
    seed=2,
)
enso_config = ReservoirConfig(
    dataset_name="ENSO",
    connectivity="opt_nn_tfi",
    memory_per_input=3,
    tau=0.1,
    seed=2,
)

lorenz_result = evaluate_configuration(lorenz, lorenz_config)
enso_result = evaluate_configuration(enso, enso_config)

fig = draw_prediction_panel("Lorenz-63", lorenz_result["forecast"]["targets"], lorenz_result["forecast"]["predictions"])
save_figure(fig, ARTIFACTS / "lorenz_predictions.png")
fig.show()

fig = draw_prediction_panel("ENSO", enso_result["forecast"]["targets"], enso_result["forecast"]["predictions"])
save_figure(fig, ARTIFACTS / "enso_predictions.png")
fig.show()

summary = [
    ("Lorenz-63", lorenz.metadata["paper_mse"], lorenz_result["baseline_mse"], lorenz_result["forecast"]["mse"]),
    ("ENSO", enso.metadata["paper_mse"], enso_result["baseline_mse"], enso_result["forecast"]["mse"]),
]

print("dataset | paper_mse | local_baseline | qrc_sim_mse")
for row in summary:
    print(f"{row[0]:10s} | {row[1]:.4f} | {row[2]:.4f} | {row[3]:.4f}")

print("\nCorrectness check")
print("- Lorenz reproduction beats the local baseline:", lorenz_result["forecast"]["mse"] < lorenz_result["baseline_mse"])
print("- ENSO reproduction beats the local baseline  :", enso_result["forecast"]["mse"] < enso_result["baseline_mse"])


**Reflection**

- Correct if the reproduced MTS-QRC forecast improves on the baseline for both systems.
- It is not necessary for the local MSE to equal the paper's exact value bit-by-bit; what matters academically is that the **same method, same qualitative ranking, and same dataset logic** are executable inside the toolbox.


## Stage 5. Rebuild the hardware-noise discussion with a local proxy

The paper's most interesting hardware claim is that:
- **Lorenz-63** behaves as expected under hardware noise,
- but **ENSO** can improve on noisy hardware.

We cannot submit to IBM hardware from this notebook, so we use a shot-and-depolarizing proxy and compare the singular-value structure of the resulting feature matrices.


In [ ]:
lorenz_proxy = hardware_proxy_forecast(lorenz, lorenz_config, shots=4096, depolarizing=0.02, seed=4)
enso_proxy = hardware_proxy_forecast(enso, enso_config, shots=4096, depolarizing=0.02, seed=4)

lorenz_ideal_svd = explained_variance(
    lorenz_result["probabilities"][: int(0.8 * len(lorenz_result["probabilities"]))],
    lorenz_config.n_qubits,
)
enso_ideal_svd = explained_variance(
    enso_result["probabilities"][: int(0.8 * len(enso_result["probabilities"]))],
    enso_config.n_qubits,
)

fig = draw_svd_panel(lorenz_ideal_svd, lorenz_proxy["svd"], "Lorenz-63")
save_figure(fig, ARTIFACTS / "lorenz_svd_proxy.png")
fig.show()

fig = draw_svd_panel(enso_ideal_svd, enso_proxy["svd"], "ENSO")
save_figure(fig, ARTIFACTS / "enso_svd_proxy.png")
fig.show()

print("Lorenz ideal MSE :", lorenz_result["forecast"]["mse"])
print("Lorenz proxy MSE :", lorenz_proxy["forecast"]["mse"])
print("Lorenz ranks     :", lorenz_ideal_svd["effective_rank"], "->", lorenz_proxy["svd"]["effective_rank"])
print()
print("ENSO ideal MSE   :", enso_result["forecast"]["mse"])
print("ENSO proxy MSE   :", enso_proxy["forecast"]["mse"])
print("ENSO ranks       :", enso_ideal_svd["effective_rank"], "->", enso_proxy["svd"]["effective_rank"])

print("\nCorrectness check")
print("- Lorenz proxy does not improve over ideal:", lorenz_proxy["forecast"]["mse"] >= lorenz_result["forecast"]["mse"])
print("- ENSO proxy stays in the same performance band:", enso_proxy["forecast"]["mse"] <= 1.5 * enso_result["forecast"]["mse"])


**Reflection**

- Correct if the proxy reproduces the paper's asymmetry: Lorenz is noise-sensitive, ENSO is comparatively noise-tolerant and may even benefit from variance compaction.
- If that asymmetry appears, then `qrc_sim` is doing something academically meaningful: it allows hypothesis testing about **feature geometry under hardware noise**.


## Closing Audit

This notebook supports the claim that **`2602.03522v1` serves as an academic tool** because it enables us to:
- translate a published QRC architecture into executable code inside the toolbox,
- inspect forecasting behavior, hyperparameter sensitivity, and resource scaling,
- and run article-oriented follow-up analyses such as singular-value diagnostics under hardware proxies.

In short, `qrc_sim` is acting here as a **research instrument**: it does not merely visualize concepts, it supports a reproducible experimental argument around a concrete paper.
